# הזמנת מלון עם Middleware לחברי עדיפות

מחברת זו מדגימה **middleware מבוסס פונקציות** באמצעות Microsoft Agent Framework. אנו בונים על דוגמת זרימת העבודה המותנית על ידי הוספת שכבת middleware שנותנת לחברי עדיפות זיכיונות מיוחדים.

## מה תלמדו:
1. **Middleware מבוסס פונקציות**: ליירט ולשנות תוצאות פונקציות
2. **גישה להקשר**: קריאה ושינוי של `context.result` לאחר ביצוע
3. **יישום לוגיקת עסקית**: הטבות לחברי עדיפות
4. **החלפת תוצאה**: שינוי תוצאות הפונקציה בהתבסס על סטטוס המשתמש
5. **אותה זרימת עבודה, תוצאות שונות**: שינויים בהתנהגות מונעים על ידי middleware

## ארכיטקטורת זרימת עבודה עם Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## הבדל מרכזי מזרימת עבודה מותנית:

**ללא Middleware** (14-conditional-workflow.ipynb):
- בפריז אין חדרים → העברה ל־alternative_agent

**עם Middleware** (מחברת זו):
- משתמש רגיל + פריז → אין חדרים → העברה ל־alternative_agent
- משתמש בעל עדיפות + פריז → 🌟 Middleware מחליף! → חדרים זמינים → העברה ל־booking_agent

## דרישות מוקדמות:
- Microsoft Agent Framework מותקן
- הבנה של זרימות עבודה מותנות (ראה 14-conditional-workflow.ipynb)
- טוקן GitHub או מפתח OpenAI API
- הבנה בסיסית של דפוסי middleware


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## שלב 1: הגדרת מודלים של Pydantic עבור פלטים מבניים

מודלים אלה מגדירים את ה**סכימה** שהסוכנים יחזירו. הוספנו שדה `priority_override` כדי לעקוב מתי שכבת התשתית משנה את תוצאת הזמינות.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## שלב 2: הגדרת מסד נתוני חברי עדיפות

בהדגמה זו, אנו נמדל מסד נתוני חברות עדיפות. בסביבה פרודקשן, זה היה שואל מסד נתונים אמתי או API.

**חברי עדיפות:**
- `alice@example.com` - חבר VIP
- `bob@example.com` - חבר פרימיום  
- `priority_user` - חשבון בדיקה


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## שלב 3: צור את כלי הזמנת המלון

כמו בתהליך העבודה המותנה, אך עכשיו הוא ייחסם על ידי תווך באמצע!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## שלב 4: 🌟 צור Middleware לבדיקת עדיפות (התכונה המרכזית!)

זו ה**פונקציונליות המרכזית** של המחברת הזו. ה-middleware:

1. **נתפס** את הקריאה לפונקציית hotel_booking  
2. **מבצע** את הפונקציה באופן רגיל על ידי קריאה ל-`next(context)`  
3. **בוחן** את התוצאה ב-`context.result`  
4. **משנה** את התוצאה אם המשתמש בעל עדיפות ואין חדרים זמינים  
5. **מחזיר** את התוצאה שהשתנתה לסוכן  

**דפוס מפתח:**  
```python
async def my_middleware(context, next):
    await next(context)  # הפעל פונקציה
    # עכשיו ה- context.result מכיל את פלט הפונקציה
    if some_condition:
        context.result = new_value  # העלה!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## שלב 5: הגדרת פונקציות תנאי לניתוב

אותן פונקציות תנאי כמו בזרימת העבודה המותנית - הן בודקות את הפלט המובנה כדי לקבוע את הניתוב.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## שלב 6: יצירת מפעיל תצוגה מותאם אישית

אותו מפעיל כמו קודם - מציג את פלט התהליך הסופי.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## שלב 7: טען משתני סביבה

הגדר את לקוח ה-LLM (GitHub Models או OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## שלב 8: יצירת סוכני בינה מלאכותית עם מִתווך

**הבדל מרכזי:** כשאנחנו יוצרים את ה-availability_agent, אנחנו מעבירים את הפרמטר `middleware`!

כך אנחנו מזריקים את ה-priority_check_middleware לצינור קריאת הפונקציות של הסוכן.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## שלב 9: בניית ה-Workflow

אותה מבנה Workflow כמו קודם - ניתוב מותנה על בסיס זמינות.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## שלב 10: מקרה מבחן 1 - משתמש רגיל בפריז (ללא ביטול) 

משתמש רגיל מנסה להזמין לפריז → אין חדרים → מפנה ל-alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## שלב 11: מקרה מבחן 2 - 🌟 משתמש בעדיפות בפריז (עם ביטול!)

חבר בעדיפות מנסה להזמין פריז → אין חדרים בהתחלה → 🌟 המידלוור מבטל! → מפנה לסוכן הזמנות

**זו ההדגמה המרכזית לכוח המידלוור!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## שלב 12: מקרה מבחן 3 - משתמש עדיפות בסטוקהולם (כבר זמין)

משתמש עדיפות מנסה את סטוקהולם → חדרים זמינים → אין צורך באוברייד → מנותב ל-booking_agent

זה מראה שה-middleware פועל רק כשצריך!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## עיקרי הדברים ומושגי מִידְלְוֵר (Middleware)

### ✅ מה שלמדתם:

#### **1. תבנית מִידְלְוֵר מבוססת פונקציות**

המִידְלְוֵר חוסם קריאות לפונקציות באמצעות פונקציית async פשוטה:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # לפני ביצוע הפונקציה
    print("Intercepting...")
    
    # להפעיל את הפונקציה
    await next(context)
    
    # לאחר ביצוע הפונקציה - לבדוק את התוצאה
    if context.result:
        # לשנות את התוצאה במידת הצורך
        context.result = modified_value
```

#### **2. גישה להקשר וביטול תוצאה**

- `context.function` - גישה לפונקציה שמוזמנת
- `context.arguments` - קריאת ארגומנטים לפונקציה
- `context.kwargs` - גישה לפרמטרים נוספים
- `await next(context)` - ביצוע הפונקציה
- `context.result` - קריאה/שינוי של תוצאת הפונקציה

#### **3. יישום לוגיקת עסק**

המִידְלְוֵר שלנו מממש הטבות לחברי עדיפות:
- **משתמשים רגילים**: ללא שינויים, זרימת עבודה סטנדרטית
- **משתמשים מועדפים**: מעלים "אין זמינות" ל-"זמין"
- **לוגיקה מותנית**: בודקת רק כשנדרש

#### **4. אותה זרימת עבודה, תוצאות שונות**

הכוח של המִידְלְוֵר:
- ✅ אין שינוי במבנה זרימת העבודה
- ✅ אין שינוי בפונקציית הכלי
- ✅ אין שינוי בלוגיקה של ניתוב מותנה
- ✅ רק מִידְלְוֵר → התנהגות שונה!

### 🚀 יישומים מעשיים:

1. **תכונות VIP/פרימיום**
   - ביטול הגבלות קצב למשתמשים פרימיום
   - הענקת גישה מועדפת למשאבים
   - פתיחת תכונות פרימיום דינמית

2. **בדיקות A/B**
   - ניתוב משתמשים למימושים שונים
   - בדיקת תכונות חדשות עם משתמשים ספציפיים
   - הפצת תכונות בהדרגה

3. **אבטחה וציות**
   - ביקורת קריאות לפונקציות
   - חסימת פעולות רגישות
   - אכיפת חוקי עסק

4. **אופטימיזציית ביצועים**
   - מטמון תוצאות למשתמשים מסוימים
   - דילוג על פעולות יקרות כשאפשרי
   - הקצאת משאבים דינמית

5. **טיפול בשגיאות וניסיון חוזר**
   - תפיסת שגיאות וטיפול נאות
   - יישום לוגיקת ניסיון חוזר
   - חזרה למימושים חלופיים

6. **רישום ומעקב**
   - מעקב אחרי זמני ביצוע פונקציות
   - רישום פרמטרים ותוצאות
   - ניטור דפוסי שימוש

### 🔑 הבדלים מרכזיים לעומת דקורטורים:

| תכונה | דקורטור | מִידְלְוֵר |
|-------|---------|----------|
| **טווח** | פונקציה בודדת | כל הפונקציות בסוכן |
| **גמישות** | קבועה בהגדרה | דינמי בזמן ריצה |
| **הקשר** | מוגבל | הקשר מלא של הסוכן |
| **הרכבה** | דקורטורים מרובים | צינור מִידְלְוֵר |
| **הכרה בסוכן** | לא | כן (גישה למצב הסוכן) |

### 📚 מתי להשתמש במִידְלְוֵר:

✅ **השתמשו במִידְלְוֵר כאשר:**
- צריך לשנות התנהגות בהתבסס על מצב משתמש/סשן
- רוצים ליישם לוגיקה על פונקציות מרובות
- דרושה גישה להקשר ברמת הסוכן
- מממשים דאגות חוצות (לוגינג, אימות וכו׳)

❌ **אל תשתמשו במִידְלְוֵר כאשר:**
- וולידציה פשוטה של קלט (השתמשו בפיידנטיק)
- לוגיקה ספציפית לפונקציה (תשאירו בתוך הפונקציה)
- שינויים חד פעמיים (פשוט תשנו את הפונקציה)

### 🎓 תבניות מתקדמות:

```python
# כמה תיווכים (סדר הביצוע חשוב!)
middleware=[
    logging_middleware,      # יומנים תחילה
    auth_middleware,         # לאחר מכן בודק את האימות
    cache_middleware,        # לאחר מכן בודק את המטמון
    rate_limit_middleware,   # לאחר מכן מגבלות תדירות
    priority_check_middleware  # לבסוף בדיקת עדיפות
]

# ביצוע תיווך מותנה
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # שינוי התוצאה
    else:
        # דלג על הביצוע לחלוטין
        context.result = cached_value
```

### 🔗 מושגים קשורים:

- **Middleware של סוכן**: חוסם קריאות של agent.run()
- **Middleware של פונקציות**: חוסם קריאות לפונקציות כלים (מה שהשתמשנו!)
- **צינור מִידְלְוֵר**: שרשרת של מִידְלְוֵר שרצים לפי סדר
- **הפצת הקשר**: העברת מצב דרך שרשרת המִידְלְוֵר


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
